# Selective Generation: When a RAG System Should Abstain — narrative notebook

This notebook imports the canonical reference `selective_generation_abstention.py` and walks the topic
movement by movement. The `.py` owns the numbers; here we read them.

The two predecessors — *faithfulness-groundedness* and *conformal-factuality* — decided **which claims to
keep** inside an answer (claim-level back-off). This topic asks the **answer-level** question their
back-off frontier set up: **whether to answer at all**. The new objects are Chow's rule, the
risk–coverage curve and its area (AURC), the achievable-vs-oracle gap, an answer-level conformal
selective-risk guarantee, and a two-stage composition with an abstain-vs-err cost model.

In [ ]:
import numpy as np
import selective_generation_abstention as sga

s = sga._setup()
scores, correct = s["scores"], s["correct"]
print(f"panel: {scores.size} answers (one per query), {sga.PANEL_NCLAIMS} claims each")
print(f"base correct-rate (faith >= {sga.FAITH_FLOOR}): {correct.mean():.3f}  "
      f"(base error {1 - correct.mean():.3f})")

## Movement 1 — Chow's rule: the cost-optimal emit/abstain decision

Emitting a wrong answer costs `c_err`; abstaining (deferring to a human analyst) costs `c_abs`. The
expected cost of emitting is `c_err · P(wrong)`, so we emit iff `c_err · P(wrong) ≤ c_abs`, i.e. iff
`P(correct) ≥ 1 − c_abs/c_err`. That cutoff is Chow's rule (1970).

In [ ]:
cutoff = sga.chow_threshold(sga.C_ERR, sga.C_ABS)
print(f"c_err={sga.C_ERR}, c_abs={sga.C_ABS}  ->  emit iff P(correct) >= {cutoff:.2f}")
emit = sga.chow_decision(scores, cutoff)
print(f"emit {int(emit.sum())} of {emit.size} answers at the Chow cutoff")
# degenerate costs recover the trivial policies
print("free abstention (c_abs=0) -> cutoff", sga.chow_threshold(5.0, 0.0), "(always abstain)")
print("c_abs >= c_err            -> cutoff", sga.chow_threshold(5.0, 5.0), "(always emit)")

## Movement 2 — the risk–coverage curve and AURC

Sweep the abstention threshold (emit the *k* most-confident answers). **Coverage** is the fraction of
queries answered; **selective risk** is the error rate among the answered. The area under this curve,
**AURC**, is the answer-level analog of average precision (area under the PR curve). The **oracle** curve
orders by true correctness; the gap from the achievable curve up to it (the *excess* AURC) is exactly the
quality of the confidence signal — it closes only as the signal's AUC → 1.

In [ ]:
g = sga.rc_gap(scores, correct)
print(f"AURC achievable {g['aurc_achievable']:.3f}  vs  oracle {g['aurc_oracle']:.3f}  "
      f"-> excess (E-AURC) {g['gap']:.3f}")
print(f"score AUC vs correct: {g['auc']:.3f}  (the gap closes to 0 as this -> 1)")
rc = sga.selective_risk_coverage(scores, correct)
print("first risk-coverage points (coverage, selective risk):")
for r in rc[12:18]:
    print(f"  coverage {r['coverage']:.3f}  selective risk {r['selective_risk']:.3f}")

## Movement 3 — an answer-level conformal selective-risk guarantee

We reuse conformal risk control (Angelopoulos et al. 2024), but applied to a **new, answer-level
monotone loss**: the *unconditional wrong-emission rate* `#{emitted ∧ wrong}/N`. It is non-increasing in
the threshold (raising the bar can only un-emit), so the imported CRC machinery consumes it directly. The
*conditional* selective risk (the risk–coverage y-axis) divides by the shrinking emitted count and is
**not** monotone — the answer-level echo of conformal-factuality's false-claim-loss vs fraction-loss.

In [ ]:
grid = np.linspace(0, 1, 51)
mono = [sga.wrong_emission_loss(scores, correct, t) for t in grid]
cond = [sga.selective_risk_loss(scores, correct, t) for t in grid]
print("wrong-emission loss monotone non-increasing:", bool(np.all(np.diff(mono) <= 1e-12)))
print("conditional selective risk non-monotone   :", bool(np.any(np.diff(cond) > 1e-9)))
t = sga.selective_conformal_threshold(scores[s["calib"]], correct[s["calib"]], sga.ALPHA)
realized = sga.wrong_emission_loss(scores[s["test"]], correct[s["test"]], t)
print(f"conformal threshold @ alpha={sga.ALPHA}: t={t:.3f}; realized test wrong-emission rate {realized:.3f}")

## Movement 4 — the two-stage composition and the cost operating point

The faithfulness hand-off: *when the certified-faithful answer is too thin to be useful, abstain rather
than emit.* So we compose two stages — claim-level conformal back-off certifies the faithful claims, then
the answer-level Chow gate abstains if the answer is too **risky** (low score) **or** too **thin** (fewer
than `MIN_CLAIMS` certified claims). The finance operating point defers the rest to a human analyst.

In [ ]:
ts = sga.two_stage_decision(s["panel"], s["conf"])
print(f"emit {int(ts['emit'].sum())} / {ts['emit'].size};  "
      f"risky-abstained {int(ts['abstain_risky'].sum())}, thin-abstained {int(ts['abstain_thin'].sum())}")
fop = sga.finance_operating_point(s["panel"], s["conf"])
print(f"defer rate {fop['defer_rate']:.3f}, residual error among emitted {fop['residual_error']:.3f}")
print(f"cost {fop['cost']:.3f}  vs  always-emit {fop['cost_always_emit']:.3f}  /  always-abstain {sga.C_ABS}")
cg = sga.calibration_gap(scores, correct)
print(f"calibration gap of the score: ECE {cg['ece']:.3f}, Brier {cg['brier']:.3f}  "
      f"(why realized != Bayes selective risk)")

## The distinctness thesis: answer-level ≠ claim-level

The same panel yields two different objects. The imported claim-level frontier (`abstention_frontier`)
sweeps a per-claim cut and reports retained-claim precision/coverage — units of **claims**. The new
risk–coverage curve sweeps an answer-level threshold — units of **answers / queries**. Claim-retention is
not answer-coverage; the curves have different x-grids and different areas.

In [ ]:
cv = sga.claim_vs_answer_curves(s["panel"], s["conf"], scores, correct)
print("claim-level frontier rows (imported):", len(cv["claim"]),
      "| answer-level risk-coverage rows (new):", len(cv["answer"]))
print("a claim-level row:", {k: round(cv['claim'][10][k], 3) for k in ('tau','faithfulness','coverage','retention')})
print("an answer-level row:", {k: round(cv['answer'][16][k], 3) for k in ('coverage','selective_risk')})

## Verification

Every assertion below is a pedagogical claim the topic makes — the collapse anchors (full-coverage risk =
base error; perfect judge → oracle; conformal threshold = the imported machinery), the build-and-run
guards (non-degenerate corpus; informative-but-imperfect signal; Chow interior beats both baselines), and
the distinctness test.

In [ ]:
sga._run_all()